# 심부전 환자 사망 예측 — EDA와 모델 학습 (Heart Failure)

- 데이터: 심부전 환자 299명의 임상 기록 (299행 · 13열)
- 목표: 사망 여부(DEATH_EVENT) 예측 — 이진분류
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_HeartFailure.txt

- 이 데이터의 핵심: **누수 변수(time)**
  → 넣으면 성능이 치솟지만 실전에선 못 쓴다

## 1. 불러오기

- 일반 CSV · 전부 수치형 · 결측 없음 · 매우 작음(299행)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("heart_failure_clinical_records_dataset.csv")
print(df.shape)          # (299, 13)
df.head()

## 2. 학습 전 확인 — 누수 변수 time

- 결측·인코딩은 AutoGluon이 자동
- 이 데이터의 핵심 판단은 **time 변수를 넣을 것인가**

### 2-1. time이 누수인 이유

- time = 환자 추적 기간(일)
- 사망자는 사망 시 추적이 끝남 → time이 짧다
- 즉 time에 이미 '결과'가 반영되어 있음
- 진료 시점에는 이 값을 알 수 없다 → 실전에서 못 씀

In [ ]:
# 사망자와 생존자의 time 비교 — 누수 확인
print("사망자 time 중앙값:", df[df.DEATH_EVENT==1]["time"].median())
print("생존자 time 중앙값:", df[df.DEATH_EVENT==0]["time"].median())
# 사망자가 훨씬 짧다 → time만 봐도 사망을 거의 맞힐 수 있다

## 3. EDA

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, progress_bar=False)
profile.to_file("heart_failure_eda.html")   # 브라우저에서 열기

### 3-1. 주요 변수와 사망

- ejection_fraction(좌심실 구혈률)이 낮으면 위험이 큰가
- serum_creatinine(신장 기능)이 높으면 위험이 큰가

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11,3))
df.boxplot(column="ejection_fraction", by="DEATH_EVENT", ax=ax[0])
df.boxplot(column="serum_creatinine", by="DEATH_EVENT", ax=ax[1])
plt.suptitle("")
plt.show()

## 4. 학습 — 누수 포함 vs 제외 비교

- 이 실습의 핵심: time을 넣은 모델과 뺀 모델을 비교
- 작은 데이터(299행)라 분할을 층화해서 안정적으로

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df["DEATH_EVENT"])   # 사망 비율 유지
print("train:", train_data.shape, "test:", test_data.shape)

### 4-1. (A) time 포함 — 누수 있는 모델

In [ ]:
pred_A = TabularPredictor(
    label="DEATH_EVENT",
    eval_metric="roc_auc",
).fit(train_data, presets="medium_quality", time_limit=180)

print("time 포함 성능:", pred_A.evaluate(test_data))
# 매우 높게 나올 것 — 하지만 실전에서 못 쓰는 성능

### 4-2. (B) time 제외 — 실전 모델

In [ ]:
cols_B = [c for c in df.columns if c != "time"]
pred_B = TabularPredictor(
    label="DEATH_EVENT",
    eval_metric="roc_auc",
).fit(train_data[cols_B], presets="medium_quality", time_limit=180)

print("time 제외 성능:", pred_B.evaluate(test_data[cols_B]))

### 4-3. 두 성능 비교

- A(포함)가 B(제외)보다 훨씬 높을 것
- 그 차이가 바로 '누수로 부풀려진 성능'
- **실전에서 신뢰할 것은 B**

In [ ]:
print("A (time 포함):", round(pred_A.evaluate(test_data)["roc_auc"], 3))
print("B (time 제외):", round(pred_B.evaluate(test_data[cols_B])["roc_auc"], 3))
# 차이가 클수록 time의 누수 효과가 크다는 뜻

## 5. 해석 (실전 모델 B 기준)

In [ ]:
pred_B.leaderboard(test_data[cols_B])

### 5-1. 변수 중요도

- time을 뺀 상태에서 무엇이 사망을 예측하는가
- ejection_fraction·serum_creatinine이 상위에 오는가
  (의학적 상식과 맞는지 확인)

In [ ]:
pred_B.feature_importance(test_data[cols_B])

## 정리

- time = 누수 변수 (결과가 반영된 값) → 실전에선 제외
- 누수 포함/제외 성능 차이 = 부풀려진 성능
- 작은 데이터(299행) → 층화 분할로 안정화
- 변수 중요도가 의학적 상식과 맞는지 확인
- 학습·앙상블은 TabularPredictor가 자동

- 누수 판단은 AutoGluon이 못 한다 — 데이터의 의미를 아는 사람 몫
  → "핵심은 사람"의 대표 사례